# 02 — Data Understanding

## Modelo predictivo para anticipar incumplimientos de SLA en HR Operations  
## (Predictive Model to Anticipate SLA Breaches in HR Operations)

El objetivo de este notebook es realizar una primera exploración del dataset base, entender su estructura, revisar la calidad inicial de los datos y evaluar si contiene las variables necesarias para construir un modelo predictivo de incumplimiento de SLA.

En esta fase no se entrena ningún modelo todavía. Primero necesitamos comprender los datos.

In [5]:
df = pd.read_csv("customer_support_tickets.csv")

In [10]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [12]:
# Cargar el dataset y mostrar las primeras filas para entender su estructura.
ruta_datos = "customer_support_tickets.csv"

df = pd.read_csv(ruta_datos)

df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [14]:
df.shape # Dimensiones del dataset

(8469, 17)

In [ ]:
df.columns # Nombres de columnas

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [16]:
df.info() # información general del dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

In [17]:
nulos = df.isnull().sum(). sort_values(ascending=False) # valores nulos
nulos

Customer Satisfaction Rating    5700
Time to Resolution              5700
Resolution                      5700
First Response Time             2819
Ticket Description                 0
Ticket Channel                     0
Ticket Priority                    0
Ticket Status                      0
Ticket ID                          0
Customer Name                      0
Ticket Type                        0
Date of Purchase                   0
Product Purchased                  0
Customer Gender                    0
Customer Age                       0
Customer Email                     0
Ticket Subject                     0
dtype: int64

Hay 3 columnas con muchos nulos:

Customer Satisfaction Rating
Time to Resolution (si el ticket todavía no se resolvió, no puede tener tiempo de resolución.)
Resolution
First Response Time → 2819 valores nulos

probablemente el dataset mezcla dos tipos de tickets: Cerrados y pendientes.

Para entrenar el modelo usaré solo tickets cerrados, porque necesitamos saber si cumplieron o incumplieron SLA. Es decir, necesitamos casos donde exista: Time to Resolution porque esa columna permite construir: incumplio_sla

Casos cerrados con Time to Resolution → sirven para entrenar
Casos pendientes sin Time to Resolution → no sirven para entrenar, pero podrían usarse en el futuro para predicción real.

Riesgo: posible pérdida de datos

Tenemos que calcular cuántas filas totales hay para saber cuántos casos útiles quedan.

In [18]:
df.shape

(8469, 17)

In [20]:
df["Time to Resolution"].isna().sum()

np.int64(5700)

El dataset tiene:
Casos con tiempo de resolución: 2769

Casos sin tiempo de resolución: 5700

Esto confirma lo que sospechábamos: El dataset mezcla tickets cerrados y tickets todavía pendientes. Para entrenar el modelo de Machine Learning, usaremos inicialmente los 2.769 casos con Time to Resolution disponible, porque son los únicos donde podemos saber si el caso cumplió o incumplió el SLA. No es un dataset enorme, pero es suficiente.

Decisión técnica: Crear un nuevo DataFrame solo con casos cerrados/resueltos. Pero antes necesitamos confirmar dos cosas:

Qué estados existen en Ticket Status.
Qué prioridades existen en Ticket Priority.

In [23]:
df["Ticket Status"].value_counts()

Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

Efectivamente los tickets con Time to Resolution son efectivamente tickets cerrados.

In [25]:
df["Ticket Priority"].value_counts() # Esto es importante para entender la distribución de prioridades en los tickets. Clave para definir el SLA.

Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

In [30]:
# Ahora tengo un dataset solo con casos que ya tienen tiempo de resolución. Esto es crucial para el análisis de SLA, ya que solo puedo evaluar el cumplimiento del SLA en tickets que han sido resueltos.
df_cerrados = df[df["Time to Resolution"].notna()].copy()

df_cerrados.shape

(2769, 17)

Esto confirma que los tickets con Time to Resolution son exactamente los tickets cerrados.

In [29]:
# Ver Status, Priority y Time to Resolution de los primeros 10 tickets cerrados para crear la variable objetivo: ¿Se incumplió el SLA o no? (incumplio_sla)
df_cerrados[["Ticket Status", "Ticket Priority", "Time to Resolution"]].head(10)

,Ticket Status,Ticket Priority,Time to Resolution
2,Closed,Low,2023-06-01 18:05:38
3,Closed,Low,2023-06-01 01:57:40
4,Closed,Low,2023-06-01 19:53:42
10,Closed,High,2023-05-31 23:51:49
11,Closed,High,2023-06-01 09:27:51
14,Closed,High,2023-05-31 23:08:55
16,Closed,Critical,2023-06-01 15:58:59
19,Closed,Low,2023-06-01 20:29:04
28,Closed,Critical,2023-06-01 06:03:17
29,Closed,Medium,2023-06-01 18:23:17


Decisión técnica importante

Ahora debo crear la variable objetivo: incumplio_sla Pero antes, como fecha y hora estan juntas debo convertir ambas columnas a fecha/hora y calcular la diferencia.

Nueva variable: horas_resolucion Calculo de cuántas horas pasaron entre la primera respuesta y la resolución.

## Creación del dataset de modelado

Para construir la variable objetivo `incumplio_sla`, primero se filtran únicamente los tickets cerrados, ya que solo estos casos tienen información completa sobre el tiempo de resolución.

El dataset original contiene tickets abiertos, pendientes y cerrados. Sin embargo, para entrenar un modelo supervisado necesitamos casos históricos cuyo resultado final sea conocido.

Por tanto, el dataset de modelado se construirá a partir de los tickets con `Time to Resolution` disponible.

In [31]:
df_cerrados = df[df["Time to Resolution"].notna()].copy()

df_cerrados.shape

(2769, 17)

## Convertir fechas

## Conversión de variables temporales

Las columnas `First Response Time` y `Time to Resolution` contienen información temporal. Para calcular el tiempo real de resolución, primero deben convertirse a formato datetime.

Después se calculará la diferencia entre ambas fechas para obtener la duración aproximada del caso en horas.

In [32]:
df_cerrados["First Response Time"] = pd.to_datetime(df_cerrados["First Response Time"])
df_cerrados["Time to Resolution"] = pd.to_datetime(df_cerrados["Time to Resolution"])

df_cerrados[["First Response Time", "Time to Resolution"]].head()

,First Response Time,Time to Resolution
2,2023-06-01 11:14:38,2023-06-01 18:05:38
3,2023-06-01 07:29:40,2023-06-01 01:57:40
4,2023-06-01 00:12:42,2023-06-01 19:53:42
10,2023-06-01 17:46:49,2023-05-31 23:51:49
11,2023-06-01 12:05:51,2023-06-01 09:27:51


## Calcular horas de resolución

In [33]:
df_cerrados["horas_resolucion"] = (
    df_cerrados["Time to Resolution"] - df_cerrados["First Response Time"]
).dt.total_seconds() / 3600

df_cerrados[["First Response Time", "Time to Resolution", "horas_resolucion"]].head(10)

,First Response Time,Time to Resolution,horas_resolucion
2,2023-06-01 11:14:38,2023-06-01 18:05:38,6.850000
3,2023-06-01 07:29:40,2023-06-01 01:57:40,-5.533333
4,2023-06-01 00:12:42,2023-06-01 19:53:42,19.683333
10,2023-06-01 17:46:49,2023-05-31 23:51:49,-17.916667
11,2023-06-01 12:05:51,2023-06-01 09:27:51,-2.633333
14,2023-06-01 06:22:55,2023-05-31 23:08:55,-7.233333
16,2023-06-01 19:46:59,2023-06-01 15:58:59,-3.800000
19,2023-06-01 00:46:04,2023-06-01 20:29:04,19.716667
28,2023-05-31 23:17:17,2023-06-01 06:03:17,6.766667
29,2023-06-01 00:54:17,2023-06-01 18:23:17,17.483333


Se detectaron inconsistencias temporales en algunos tickets, donde la fecha de resolución aparece antes de la fecha de primera respuesta. Estos registros no son válidos para calcular duración operativa y deben ser tratados antes de construir la variable objetivo.

Aqui hay valores negativos. Significa que, en algunas filas, el dataset dice que el ticket se resolvió antes de la primera respuesta. No tiene sentido operativo. 
El problema está en los datos: Hay registros donde Time to Resolution aparece antes que First Response Time.

Lo que hare será medir cuántos casos tienen duración negativa.

In [34]:
casos_negativos = (df_cerrados["horas_resolucion"] < 0).sum()

casos_negativos

np.int64(1365)

## Validación de consistencia temporal

Antes de construir la variable objetivo, se revisó si existían inconsistencias temporales en el cálculo de `horas_resolucion`.

La duración del caso se calculó como la diferencia entre `Time to Resolution` y `First Response Time`.

Sin embargo, algunos registros presentan valores negativos, lo que indica que la fecha de resolución aparece antes que la fecha de primera respuesta. Estos casos no son consistentes desde el punto de vista operativo y deben tratarse antes del modelado.

In [37]:
(df_cerrados["horas_resolucion"] < 0).sum()

np.int64(1365)

In [38]:
((df_cerrados["horas_resolucion"] < 0).mean() * 100).round(2)

np.float64(49.3)

In [39]:
df_cerrados["horas_resolucion"].describe()

count    2769.000000
mean       -0.057704
std         9.564112
min       -23.233333
25%        -6.933333
50%         0.166667
75%         6.483333
max        23.466667
Name: horas_resolucion, dtype: float64

aquí No vamos a eliminar los negativos, porque son demasiados.

Casos negativos: 1.365
Porcentaje negativo: 49,3%

Eso significa que casi la mitad del dataset tiene una inconsistencia temporal. Si eliminamos esos casos, perderíamos casi la mitad de los datos cerrados y el modelo quedaría debilitado.

Decisión técnica: Como el porcentaje de negativos es muy alto, haremos una corrección metodológica:

Cuando horas_resolucion sea negativa, se sumarán 24 horas para corregir posibles cruces de día o inconsistencias de fecha dentro de una ventana diaria.

Esto es más razonable que eliminar el 49,3% de los datos.

Dado que casi la mitad de los registros presentaban valores negativos en `horas_resolucion`, no se eliminaron directamente para evitar una pérdida excesiva de datos.

Al observar que los valores negativos se encontraban dentro de un rango aproximado de -24 horas, se interpretó que la inconsistencia podía deberse a errores de registro temporal o cruces de día en el dataset.

Por este motivo, se aplicó una corrección operativa: cuando la duración calculada era negativa, se sumaron 24 horas para estimar una duración positiva dentro de una ventana diaria.

## Corrección de inconsistencias temporales

Se detectó que el 49,3% de los tickets cerrados presentaban valores negativos en `horas_resolucion`.

Dado que eliminar casi la mitad de los registros reduciría demasiado el tamaño del dataset, se decidió aplicar una corrección temporal.

Como los valores negativos se encontraban dentro de un rango aproximado de -24 horas, se asumió que estas inconsistencias podían deberse a cruces de día o errores de registro de fecha.

La corrección aplicada fue:

- Si `horas_resolucion` es negativa, se suman 24 horas.
- Si `horas_resolucion` es positiva, se mantiene el valor original.

Esta corrección permite conservar los registros y trabajar con una estimación operativa razonable del tiempo de resolución.

In [40]:
df_cerrados["horas_resolucion_corregidas"] = np.where(
    df_cerrados["horas_resolucion"] < 0,
    df_cerrados["horas_resolucion"] + 24,
    df_cerrados["horas_resolucion"]
)

df_cerrados[[
    "First Response Time",
    "Time to Resolution",
    "horas_resolucion",
    "horas_resolucion_corregidas"
]].head(10)

,First Response Time,Time to Resolution,horas_resolucion,horas_resolucion_corregidas
2,2023-06-01 11:14:38,2023-06-01 18:05:38,6.850000,6.850000
3,2023-06-01 07:29:40,2023-06-01 01:57:40,-5.533333,18.466667
4,2023-06-01 00:12:42,2023-06-01 19:53:42,19.683333,19.683333
10,2023-06-01 17:46:49,2023-05-31 23:51:49,-17.916667,6.083333
11,2023-06-01 12:05:51,2023-06-01 09:27:51,-2.633333,21.366667
14,2023-06-01 06:22:55,2023-05-31 23:08:55,-7.233333,16.766667
16,2023-06-01 19:46:59,2023-06-01 15:58:59,-3.800000,20.200000
19,2023-06-01 00:46:04,2023-06-01 20:29:04,19.716667,19.716667
28,2023-05-31 23:17:17,2023-06-01 06:03:17,6.766667,6.766667
29,2023-06-01 00:54:17,2023-06-01 18:23:17,17.483333,17.483333


Verificar que ya no hay negativos:

In [41]:
(df_cerrados["horas_resolucion_corregidas"] < 0).sum()

np.int64(0)

In [42]:
df_cerrados["horas_resolucion_corregidas"].describe()

count    2769.000000
mean       11.773282
std         7.042174
min         0.000000
25%         5.333333
50%        11.616667
75%        17.950000
max        23.983333
Name: horas_resolucion_corregidas, dtype: float64

Ahora sí podemos crear el SLA esperado.

Critical	6 horas
High	12 horas
Medium	18 horas
Low	24 horas

Cambio importante: antes habíamos propuesto 8, 24, 72, 120 horas, pero ahora vemos que la duración corregida se mueve dentro de un rango diario. Por eso debemos adaptar la regla al dataset.

Si usáramos 72 o 120 horas, casi nadie incumpliría SLA y el modelo no tendría sentido.

## Crear SLA esperado: 

## Definición del SLA esperado por prioridad

Tras corregir la duración de resolución, se observa que los tiempos se distribuyen dentro de una ventana aproximada de 24 horas.

Por este motivo, se define una regla de SLA ajustada al comportamiento temporal del dataset:

- `Critical`: 6 horas.
- `High`: 12 horas.
- `Medium`: 18 horas.
- `Low`: 24 horas.

Esta regla mantiene una lógica operativa realista: cuanto mayor es la prioridad del caso, menor es el tiempo permitido para resolverlo.

In [44]:
sla_por_prioridad = {
    "Critical": 6,
    "High": 12,
    "Medium": 18,
    "Low": 24
}

df_cerrados["sla_esperado_horas"] = df_cerrados["Ticket Priority"].map(sla_por_prioridad)

df_cerrados[["Ticket Priority", "sla_esperado_horas"]].head(10)

,Ticket Priority,sla_esperado_horas
2,Low,24
3,Low,24
4,Low,24
10,High,12
11,High,12
14,High,12
16,Critical,6
19,Low,24
28,Critical,6
29,Medium,18


## Crear la variable objetivo incumplio_sla

In [45]:
df_cerrados["incumplio_sla"] = np.where(
    df_cerrados["horas_resolucion_corregidas"] > df_cerrados["sla_esperado_horas"],
    1,
    0
)

df_cerrados[[
    "Ticket Priority",
    "horas_resolucion_corregidas",
    "sla_esperado_horas",
    "incumplio_sla"
]].head(20)

,Ticket Priority,horas_resolucion_corregidas,sla_esperado_horas,incumplio_sla
2,Low,6.850000,24,0
3,Low,18.466667,24,0
4,Low,19.683333,24,0
10,High,6.083333,12,0
11,High,21.366667,12,1
14,High,16.766667,12,1
16,Critical,20.200000,6,1
19,Low,19.716667,24,0
28,Critical,6.766667,6,1
29,Medium,17.483333,18,0


## Revisar distribución del target

In [46]:
df_cerrados["incumplio_sla"].value_counts()

incumplio_sla
0    1711
1    1058
Name: count, dtype: int64

In [47]:
df_cerrados["incumplio_sla"].value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

In [48]:
df_cerrados["horas_resolucion_corregidas"].describe()

count    2769.000000
mean       11.773282
std         7.042174
min         0.000000
25%         5.333333
50%        11.616667
75%        17.950000
max        23.983333
Name: horas_resolucion_corregidas, dtype: float64

In [49]:
df_cerrados["incumplio_sla"].value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

Esto significa:

Métrica	-------------------------Interpretación

count = 2769---------------------Tenemos 2.769 tickets cerrados válidos

mean = 11.77---------------------El tiempo medio de resolución es de unas 11,8 horas

min = 0--------------------------Ya no hay tiempos negativos

50% = 11.62----------------------La mitad de los tickets se resuelve en menos de 11,6 horas

max = 23.98----------------------Los tiempos están dentro de una ventana diaria


Conclusión: La corrección temporal funcionó. Ahora la variable de duración es coherente para construir el SLA.

## Distribución de la variable objetivo

La variable objetivo `incumplio_sla` presenta la siguiente distribución:

| Valor | Significado | Porcentaje |
|---|---|---:|
| `0` | El ticket cumplió el SLA | 61,79% |
| `1` | El ticket incumplió el SLA | 38,21% |

Esta distribución es adecuada para un problema de clasificación binaria.

Aunque la clase `0` es mayoritaria, la clase `1` representa un 38,21% de los casos, por lo que existe una cantidad suficiente de ejemplos de incumplimiento para entrenar modelos de Machine Learning.

Desde el punto de vista de negocio, un 38,21% de incumplimientos representa un volumen relevante de riesgo operativo. Esto justifica el uso de un modelo predictivo para anticipar casos críticos y ayudar a priorizar la carga de trabajo.

In [51]:
# Revisar columnas finales creadas
df_cerrados.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating', 'horas_resolucion',
       'horas_resolucion_corregidas', 'sla_esperado_horas', 'incumplio_sla'],
      dtype='object')

In [52]:
# Ver una muestra final del dataset preparado
columnas_revision = [
    "Ticket ID",
    "Ticket Status",
    "Ticket Priority",
    "First Response Time",
    "Time to Resolution",
    "horas_resolucion",
    "horas_resolucion_corregidas",
    "sla_esperado_horas",
    "incumplio_sla"
]

df_cerrados[columnas_revision].head(10)

,Ticket ID,Ticket Status,Ticket Priority,First Response Time,Time to Resolution,horas_resolucion,horas_resolucion_corregidas,sla_esperado_horas,incumplio_sla
2,3,Closed,Low,2023-06-01 11:14:38,2023-06-01 18:05:38,6.850000,6.850000,24,0
3,4,Closed,Low,2023-06-01 07:29:40,2023-06-01 01:57:40,-5.533333,18.466667,24,0
4,5,Closed,Low,2023-06-01 00:12:42,2023-06-01 19:53:42,19.683333,19.683333,24,0
10,11,Closed,High,2023-06-01 17:46:49,2023-05-31 23:51:49,-17.916667,6.083333,12,0
11,12,Closed,High,2023-06-01 12:05:51,2023-06-01 09:27:51,-2.633333,21.366667,12,1
14,15,Closed,High,2023-06-01 06:22:55,2023-05-31 23:08:55,-7.233333,16.766667,12,1
16,17,Closed,Critical,2023-06-01 19:46:59,2023-06-01 15:58:59,-3.800000,20.200000,6,1
19,20,Closed,Low,2023-06-01 00:46:04,2023-06-01 20:29:04,19.716667,19.716667,24,0
28,29,Closed,Critical,2023-05-31 23:17:17,2023-06-01 06:03:17,6.766667,6.766667,6,1
29,30,Closed,Medium,2023-06-01 00:54:17,2023-06-01 18:23:17,17.483333,17.483333,18,0


## Guardado del dataset intermedio

Después de filtrar los tickets cerrados, corregir las inconsistencias temporales y construir la variable objetivo `incumplio_sla`, se guarda una primera versión intermedia del dataset.

Este archivo será utilizado en los siguientes notebooks para limpieza final, feature engineering y modelado.

In [61]:
import os

# Ruta relativa desde donde está el notebook
# Mejor opción para trabajo en equipo
ruta_salida = "../data/interim/tickets_hr_sla_interim.csv"

df_cerrados.to_csv(ruta_salida, index=False)
print(f"Dataset guardado en: {ruta_salida}")

Dataset guardado en: ../data/interim/tickets_hr_sla_interim.csv


In [62]:
df_verificacion = pd.read_csv("../data/interim/tickets_hr_sla_interim.csv")

df_verificacion.shape

(2769, 21)

In [63]:
# Verificar que la variable objetivo sigue existiendo
df_verificacion["incumplio_sla"].value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

## Conclusión del Data Understanding

En este notebook se realizó la primera exploración del dataset base y se confirmó que puede adaptarse a un caso de HR Operations.

El dataset original contenía 8.469 tickets y 17 columnas. Sin embargo, no todos los tickets estaban cerrados. Para construir un modelo supervisado, se utilizaron únicamente los 2.769 tickets con información completa de resolución.

Durante la exploración se detectaron inconsistencias temporales en el cálculo inicial de duración. Algunos registros presentaban valores negativos, lo que indicaba que la fecha de resolución aparecía antes de la fecha de primera respuesta.

Dado que estos valores negativos representaban el 49,3% de los tickets cerrados y se encontraban dentro de una ventana aproximada de 24 horas, se aplicó una corrección temporal sumando 24 horas a las duraciones negativas.

Después de esta corrección, se creó la variable `horas_resolucion_corregidas`, con una duración media de aproximadamente 11,77 horas y valores dentro de una ventana operativa coherente.

Posteriormente, se definió una regla de SLA esperado basada en la prioridad del ticket:

| Prioridad | SLA esperado |
|---|---:|
| Critical | 6 horas |
| High | 12 horas |
| Medium | 18 horas |
| Low | 24 horas |

A partir de esta regla, se creó la variable objetivo `incumplio_sla`:

| Valor | Significado |
|---|---|
| `0` | El ticket cumplió el SLA |
| `1` | El ticket incumplió el SLA |

La distribución final de la variable objetivo fue:

| Resultado | Porcentaje |
|---|---:|
| Cumplió SLA | 61,79% |
| Incumplió SLA | 38,21% |

Esta distribución es adecuada para avanzar hacia un problema de clasificación binaria supervisada.

El dataset intermedio fue guardado en:

`../data/interim/tickets_hr_sla_interim.csv`

Este archivo será la base para los siguientes pasos del proyecto: limpieza final, feature engineering, separación train/test y aplicación de modelos de Machine Learning.